# An Empirical Analysis of Probability Calibration in Imbalanced Financial Tasks

## Experimentation Pipeline

In [ ]:
!pip install betacal
!pip install ucimlrepo

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    matthews_corrcoef,
    roc_auc_score,
)
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, PowerTransformer, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.frozen import FrozenEstimator
import xgboost as xgb
from ucimlrepo import fetch_ucirepo
from betacal import BetaCalibration
import matplotlib.pyplot as plt
import seaborn as sns


## Data Loading

One loader function for each dataset and a helper to automate loading by name

In [ ]:
def load_german_credit():
    """Load Statlog (German Credit) - Hofmann 1994, UCI. 1000 instances, 20 features"""
    data = fetch_ucirepo(id=144)
    X = data.data.features
    y = data.data.targets.iloc[:, 0]
    # Raw target values: 1=Good, 2=Bad
    y = (y == 2).astype(int) # Bad(default)=1(positive)
    return X, y

def get_german_credit_columns_info():
  numeric = ['Attribute2', 'Attribute5', 'Attribute8', 'Attribute11', 'Attribute13', 'Attribute16', 'Attribute18']
  nominal = ['Attribute4', 'Attribute9', 'Attribute10', 'Attribute14', 'Attribute15', 'Attribute19', 'Attribute20']
  ordinal = ['Attribute1', 'Attribute3', 'Attribute6', 'Attribute7', 'Attribute12', 'Attribute17']
  ordinal_categories = [
      ["A14", "A11", "A12", "A13"], # Attribute1
      ["A30", "A31", "A32", "A33", "A34"], # Attribute3
      ["A65", "A61", "A62", "A63", "A64"], # Attribute6
      ["A71", "A72", "A73", "A74", "A75"], # Attribute7
      ["A124", "A123", "A122", "A121"], # Attribute12
      ["A171", "A172", "A173", "A174"], # Attribute17
  ]
  return numeric, nominal, ordinal, ordinal_categories

In [ ]:
def load_default_credit_card():
    """Load Default of Credit Card Clients (Taiwan) - Yeh 2009, UCI. 30k instances, 23 features"""
    data = fetch_ucirepo(id=350)
    X = data.data.features
    y = data.data.targets.iloc[:, 0]
    return X, y

def get_default_credit_card_columns_info():
    numeric = ["X1", "X5", "X12", "X13", "X14", "X15", "X16", "X17", "X18", "X19", "X20", "X21", "X22", "X23"]
    nominal = ["X2", "X3", "X4"]
    ordinal = ["X6", "X7", "X8", "X9", "X10", "X11"]
    ordinal_categories = [
        [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9], # X6
        [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9], # X7
        [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9], # X8
        [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9], # X9
        [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9], # X10
        [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9], # X11
    ]
    return numeric, nominal, ordinal, ordinal_categories


In [ ]:
def load_bank_marketing():
    """Load Bank Marketing - Moro et al. 2014, UCI id=222. 45k+ instances, 16 features"""
    data = fetch_ucirepo(id=222)
    X = data.data.features
    y = data.data.targets.iloc[:, 0]
    # Raw target values: "yes"/"no"
    y = (y.astype(str).str.lower() == "yes").astype(int)
    return X, y


def get_bank_marketing_columns_info():
    numeric = ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous']
    nominal = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'day_of_week', 'month', 'poutcome']
    ordinal = []
    ordinal_categories = []
    return numeric, nominal, ordinal, ordinal_categories

# Data Split and Preprocessing

In [ ]:
def split_dataset(X, y):
    """ trainig 60% - calibration 25% - test 15% """
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.15, stratify=y, random_state=42
    )
    X_train, X_calib, y_train, y_calib = train_test_split(
        X_temp, y_temp, test_size=0.2941, stratify=y_temp, random_state=42
    )
    return X_train, X_calib, X_test, y_train, y_calib, y_test

In [ ]:
def get_preprocessor(columns_info, X_train):
    numeric_cols, nominal_cols, ordinal_cols, ordinal_categories = columns_info

    skew = X_train[numeric_cols].skew().abs()
    skewed_cols = skew[skew > 0.5].index.tolist()
    non_skewed_cols = [c for c in numeric_cols if c not in skewed_cols]

    skewed_numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("power", PowerTransformer(method="yeo-johnson")),
        ("scaler", StandardScaler())
    ])
    normal_numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    nominal_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    ordinal_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(categories=ordinal_categories))
    ])

    preprocessor = ColumnTransformer([
        ("num_skewed", skewed_numeric_pipeline, skewed_cols),
        ("num_normal", normal_numeric_pipeline, non_skewed_cols),
        ("ord", ordinal_pipeline, ordinal_cols),
        ("nom", nominal_pipeline, nominal_cols)
    ])

    return preprocessor

# Training

In [ ]:
RANDOM_STATE = 42

In [ ]:
def get_classification_models(y_train):
  """Model definitions with grid search params (optimizing AUC-ROC)
  y_train: training labels, used to compute scale_pos_weight for XGBoost
  """
  scale_pos_weight = [1]
  if y_train is not None:
    neg, pos = np.sum(y_train == 0), np.sum(y_train == 1)
    scale_pos_weight = [1, neg / max(pos, 1)]

  return {
    "Logistic Regression": {
        "model": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE,),
        "param_grid": [
          {
            "solver": ["liblinear"],
            "penalty": ["l1", "l2"],
            "C": [0.001, 0.01, 0.1, 1, 10, 100],
          },
          {
            "solver": ["lbfgs"],
            "penalty": ["l2"],
            "C": [0.001, 0.01, 0.1, 1, 10, 100],
          },
          {
            "solver": ["saga"],
            "penalty": ["elasticnet"],
            "C": [0.001, 0.01, 0.1, 1, 10, 100],
            "l1_ratio": [0, 0.5, 1],
          },
      ],
    },

    "Random Forest": {
        "model": RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        "param_grid": {
          "n_estimators": [100, 200, 500],
          "max_depth": [10, 15, None],
          "max_features": ["sqrt", "log2"],
          "min_samples_split": [2, 5, 10],
          "min_samples_leaf": [1, 2, 5],
          "bootstrap": [True, False],
        },
    },

    "XGBoost": {
        "model": xgb.XGBClassifier(use_label_encoder=False, random_state=RANDOM_STATE),
        "param_grid": {
          "learning_rate": [0.01, 0.05, 0.1, 0.2],
          "max_depth": [3, 5, 7],
          "n_estimators": [100, 200, 500],
          "subsample": [0.8, 1.0],
          "colsample_bytree": [0.8, 1.0],
          "gamma": [0, 0.1, 0.3],
          "scale_pos_weight": scale_pos_weight,
          "reg_alpha": [0, 0.01, 0.1],
          "reg_lambda": [1, 1.5, 2],
        },
    }
  }

In [ ]:
def train_model(model, param_grid, X_train, y_train):
  gs = RandomizedSearchCV(
    model,
    param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1,
    random_state=RANDOM_STATE
  )
  gs.fit(X_train, y_train)
  return gs.best_estimator_, gs.best_params_

# Calibration

In [ ]:
def compute_calibrated_probabilities(model, X_cal, y_cal, X_test):
  """ Return the calibrated predicted probabilities for X_test """
  y_cal = np.asarray(y_cal).ravel()

  # Platt Scaling
  platt = CalibratedClassifierCV(FrozenEstimator(model), method="sigmoid")
  platt.fit(X_cal, y_cal)
  platt_pred_proba = platt.predict_proba(X_test)[:, 1]

  # Isotonic Regression
  isotonic = CalibratedClassifierCV(FrozenEstimator(model), method="isotonic")
  isotonic.fit(X_cal, y_cal)
  isotonic_pred_proba = isotonic.predict_proba(X_test)[:, 1]

  # Beta
  beta_probs_cal = np.clip(model.predict_proba(X_cal)[:, 1], 1e-6, 1 - 1e-6)
  beta_bc = BetaCalibration(parameters="abm")
  beta_bc.fit(beta_probs_cal.reshape(-1, 1), y_cal)
  beta_probs_pred = np.clip(model.predict_proba(X_test)[:, 1], 1e-6, 1 - 1e-6)
  beta_pred_proba = beta_bc.predict(beta_probs_pred.reshape(-1, 1))

  return platt_pred_proba, isotonic_pred_proba, beta_pred_proba

# Evaluation

In [ ]:
def calculate_brier_components(y_true, y_proba, n_bins=10):
  """Murphy (1973) Brier Score decomposition: BS = MCB - DSC + UNC"""
  y_true, y_proba = np.asarray(y_true), np.asarray(y_proba)
  base_rate = np.mean(y_true)
  unc       = base_rate * (1 - base_rate)   # UNC — dataset constant
  # Quantile bin edges — deduplicate for discrete ensemble outputs
  edges = np.quantile(y_proba, np.linspace(0, 1, n_bins + 1))
  edges = np.unique(edges)
  edges[-1] += 1e-9
  n_actual_bins = len(edges) - 1
  binids     = np.clip(np.digitize(y_proba, edges) - 1, 0, n_actual_bins - 1)
  bin_counts = np.bincount(binids, minlength=n_actual_bins)
  prob_true = np.array([
    np.mean(y_true[binids == b]) if bin_counts[b] > 0 else 0.0
    for b in range(n_actual_bins)
  ])
  prob_pred = np.array([
    np.mean(y_proba[binids == b]) if bin_counts[b] > 0 else 0.0
    for b in range(n_actual_bins)
  ])
  N = len(y_true)
  weights = bin_counts / N
  mcb = np.sum(weights * (prob_true - prob_pred) ** 2)
  dsc = np.sum(weights * (prob_true - base_rate) ** 2)
  return mcb, dsc, unc

In [ ]:
def compute_ece_mce(y_true, y_pred_prob, num_bins=10):
  """Expected Calibration Error (ECE) and Maximum Calibration Error (MCE)"""
  y_true      = np.asarray(y_true)
  y_pred_prob = np.asarray(y_pred_prob)
  bin_edges   = np.linspace(0, 1, num_bins + 1)
  bin_indices = np.clip(
    np.digitize(y_pred_prob, bin_edges) - 1, 0, num_bins - 1
  )
  ece, mce = 0.0, 0.0
  for b in range(num_bins):
    mask = bin_indices == b
    if mask.sum() == 0:
      continue
    avg_conf = np.mean(y_pred_prob[mask])
    obs_freq = np.mean(y_true[mask])
    bin_error = abs(avg_conf - obs_freq)
    ece += (mask.sum() / len(y_true)) * bin_error
    mce = max(mce, bin_error)

  return ece, mce

In [ ]:
def compute_metrics(y_test, y_test_proba):
  ece, mce = compute_ece_mce(y_test, y_test_proba, num_bins=10)
  mcb, dsc, unc = calculate_brier_components(y_test, y_test_proba, n_bins=10)
  bs = brier_score_loss(y_test, y_test_proba)

  # Sanity check: MCB - DSC + UNC should approximate BS
  # Differences > 0.001 indicate a binning problem
  bs_reconstructed = mcb - dsc + unc
  if abs(bs_reconstructed - bs) > 0.001:
    warnings.warn(
      f"Brier decomposition mismatch: BS={bs:.4f}, "
      f"MCB-DSC+UNC={bs_reconstructed:.4f}, "
      f"diff={abs(bs_reconstructed - bs):.4f}"
    )

  return {
    "auc_roc": roc_auc_score(y_test, y_test_proba),
    "log_loss": log_loss(y_test, y_test_proba),
    "brier_score": bs,
    "brier_mcb": mcb,
    "brier_dsc": dsc,
    "ece": ece,
    "mce": mce,
  }

# Run Pipeline

In [ ]:
def run_pipeline_for_dataset(dataset_name, data_loader, columns_info_resolver):
  # Data Loading
  X, y = data_loader()

  # 3-way data split
  X_train, X_cal, X_test, y_train, y_cal, y_test = split_dataset(X, y)

  # preprocessing
  preprocessor = get_preprocessor(columns_info_resolver(), X_train)
  X_train = preprocessor.fit_transform(X_train)
  X_cal = preprocessor.transform(X_cal)
  X_test = preprocessor.transform(X_test)

  # pipeline
  metrics_list = []
  df_y = pd.DataFrame()
  models = get_classification_models(y_train)
  for model_name, model_info in models.items():
    # training
    best_model, best_hyperparams = train_model(
      model_info['model'],
      model_info['param_grid'],
      X_train,
      y_train
    )

    # raw probabilities
    y_cal_proba = best_model.predict_proba(X_cal)[:, 1]
    y_test_proba = best_model.predict_proba(X_test)[:, 1]

    # calibration
    calib_methods_proba = compute_calibrated_probabilities(best_model, X_cal, y_cal, X_test)
    y_platt_proba, y_isotonic_proba, y_beta_proba = calib_methods_proba

    # evaluation
    base_results = compute_metrics(y_test, y_test_proba)
    platt_results = compute_metrics(y_test, y_platt_proba)
    isotonic_results = compute_metrics(y_test, y_isotonic_proba)
    beta_results = compute_metrics(y_test, y_beta_proba)

    # save metrics
    model_label = {'model': model_name, 'hyperparameters': str(best_hyperparams)}
    metrics_list.append({**model_label, 'calibration method': 'base', **base_results})
    metrics_list.append({**model_label, 'calibration method': 'platt', **platt_results})
    metrics_list.append({**model_label, 'calibration method': 'isotonic', **isotonic_results})
    metrics_list.append({**model_label, 'calibration method': 'beta', **beta_results})

    df_y[f"{model_name} y_test_proba"] = np.asarray(y_test_proba)
    df_y[f"{model_name} y_platt_proba"] = np.asarray(y_platt_proba)
    df_y[f"{model_name} y_isotonic_proba"] = np.asarray(y_isotonic_proba)
    df_y[f"{model_name} y_beta_proba"] = np.asarray(y_beta_proba)


  # save results to csv
  output_dir = Path("csv_results")
  output_dir.mkdir(exist_ok=True)

  df_metrics = pd.DataFrame(metrics_list)
  df_metrics.to_csv(output_dir / f"{dataset_name}-metrics_results.csv", index=False)

  df_y['y_test'] = np.asarray(y_test)
  df_y.to_csv(output_dir / f"{dataset_name}-y_results.csv", index=False)

In [ ]:
""" Main """
loaders = {
  "German Credit": (load_german_credit, get_german_credit_columns_info),
  "Credit Card Default": (load_default_credit_card, get_default_credit_card_columns_info),
  "Bank Term Deposit": (load_bank_marketing, get_bank_marketing_columns_info),
}

for dataset_name, (data_loader, columns_info_resolver) in loaders.items():
  run_pipeline_for_dataset(dataset_name, data_loader, columns_info_resolver)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Fitting 5 folds for each of 10 candidates, totalling 50 fits
